In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

def run_shap_analysis(xgb_model, X_test, y_test):
    """Calculates global SHAP weights and dissects local predictions for edge cases."""
    # 1. Native Feature Importance Baseline
    importances = xgb_model.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    plt.figure(figsize=(10, 4))
    sns.barplot(x=importances[indices[:10]], y=X_test.columns[indices[:10]], palette="rocket")
    plt.title("Top 10 Built-In Feature Importances (Gain)")
    plt.tight_layout()
    plt.show()
    
    # 2. Structural SHAP Computation
    explainer = shap.TreeExplainer(xgb_model)
    shap_values = explainer(X_test)
    
    plt.figure(figsize=(10, 5))
    shap.summary_plot(shap_values, X_test, max_display=10, show=False)
    plt.title("SHAP Global Summary Plot", fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # 3. Local Operational Edge Case Extraction
    test_predictions = xgb_model.predict(X_test)
    y_test_arr = y_test.values
    
    true_positives = np.where((y_test_arr == 1) & (test_predictions == 1))[0]
    false_positives = np.where((y_test_arr == 0) & (test_predictions == 1))[0]
    false_negatives = np.where((y_test_arr == 1) & (test_predictions == 0))[0]
    
    def log_local_drivers(case_indices, label_str):
        if len(case_indices) == 0: 
            print(f"\n[SHAP Breakdown: {label_str}] No cases matching signature found in test segment.")
            return
        target_idx = case_indices[0]
        local_shap = shap_values[target_idx].values
        ranked_features = np.argsort(np.abs(local_shap))[::-1][:3]
        
        print(f"\n[SHAP Breakdown: {label_str}] (Test Row Index: {target_idx})")
        for rank, f_idx in enumerate(ranked_features):
            name = X_test.columns[f_idx]
            val = X_test.iloc[target_idx, f_idx]
            impact = local_shap[f_idx]
            print(f"   Rank {rank+1} -> {name:<35} | Scaled Value: {val:>6.2f} | SHAP Force: {impact:>6.2f}")

    log_local_drivers(true_positives, "True Positive (Correct Catch)")
    log_local_drivers(false_positives, "False Positive (User Insult / False Alarm)")
    log_local_drivers(false_negatives, "False Negative (Slipped Leakage)")